In [1]:
from pathlib import Path
import xarray as xr
import ecco_v4_py as ecco
from xgcm import Grid
import gsw
import matplotlib.pyplot as plt
import cmocean
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import cartopy.crs as ccrs
from os.path import join,expanduser,exists,split
import glob

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
ECCO_version = 'v4r4'
#ECCO_version = 'v4r5'

if ECCO_version == 'v4r4':
    # v4r4
    ECCO_dir = Path.home() / 'share_disk3/ECCO_products/Version4/Release4'
    grid_file_path = glob.glob(join(ECCO_dir,'*GEOMETRY*','*GEOMETRY*.nc'))[0]
    TS_file_paths           = join(ECCO_dir,'*TEMP_SALINITY*MONTHLY*',f'*.nc')
else: # by default
    ## v4r5
    ECCO_version = 'v4r5'
    ECCO_grid = Path.home() / 'share_disk3/ECCO_products/Version4/Release5/netcdf/native/grid/'
    ECCO_dir = Path.home() / 'share_disk3/ECCO_products/Version4/Release5/netcdf/native/monthly/'
    grid_file_path = glob.glob(join(ECCO_grid,'*GEOMETRY*.nc'))[0]
    TS_file_paths           = join(ECCO_dir,'*TEMPERATURE_SALINITY',f'*.nc')
print(ECCO_version)

v4r4


In [4]:
## Load the model grid
ecco_grid = xr.open_mfdataset(grid_file_path,\
                              chunks={'k':50,'tile':13,'j':90,'j_g':90,'i':90,'i_g':90})

In [5]:
## Create a dataset of monthly advective and diffusive temperature fluxes, 1992-2017
ecco_vars_ts = xr.open_mfdataset(TS_file_paths,\
                              chunks={'k':50,'tile':13,'j':90,'j_g':90,'i':90,'i_g':90},\
                              parallel=True,data_vars='minimal',coords='minimal',compat='override')

In [6]:
## Merge the ecco_grid with the ecco_vars to make the ecco_ds
ecco_ds = xr.merge((ecco_grid , ecco_vars_ts),compat='override')

In [7]:
# define the metrics
ecco_ds['drW'] = ecco_ds.hFacW * ecco_ds.drF #vertical cell size at u point
ecco_ds['drS'] = ecco_ds.hFacS * ecco_ds.drF #vertical cell size at v point
ecco_ds['drV'] = ecco_ds.rA * ecco_ds.drF # volume at centre and k
ecco_ds['drVg'] = ecco_ds.rAz * ecco_ds.drF # volume at q and k
ecco_ds['drVw'] = ecco_ds.rAw * ecco_ds.drF # volume at q and k
ecco_ds['drVs'] = ecco_ds.rAs * ecco_ds.drF # volume at q and k

metrics = { ('X',): ['dxC', 'dxG'], # X distances
    ('Y',): ['dyC', 'dyG'], # Y distances
    ('Z',): ['drW', 'drS', 'drC','drF'], # Z distances
    ('X', 'Y'): ['rA', 'rAz', 'rAs', 'rAw'], #Areas
    ('X', 'Y','Z'): ['drV','drVg','drVs','drVw']} #volumes

grid = Grid(ecco_ds,periodic=False, metrics=metrics)

In [8]:
globmask = ecco_ds.hFacC.where(ecco_ds.hFacC==0,1)
total_volume = grid.integrate(globmask,['X','Y','Z']).sum('tile').compute()
centre_volume = grid.integrate(ecco_ds.Z*globmask ,['X','Y','Z']).sum('tile') / total_volume
profile_volume = grid.integrate(ecco_ds.hFacC,['X','Y']).sum('tile').compute()
local_depth = grid.integrate(globmask,['Z'])

g = 9.81
rho0 = 1026

In [9]:
H2 = grid.integrate(ecco_ds.maskC.where(ecco_ds.Z>-2000),'Z') #Integrating the maskC field over the vertical dimension Z
maskC_2000 = ecco_ds.maskC.where(ecco_ds.Z>-2000).where(H2==np.max(H2))
volume_2000 = grid.integrate(maskC_2000,['X','Y','Z']).sum('tile').compute() 

h0 = gsw.dynamic_enthalpy(ecco_ds.SALT, ecco_ds.THETA,-ecco_ds.Z)
h0_zv = gsw.dynamic_enthalpy(ecco_ds.SALT, ecco_ds.THETA,H2/2)
h_zv = h0 - h0_zv

mean_THETA = grid.integrate(ecco_ds.THETA * maskC_2000,['X','Y','Z']).sum('tile').compute() / volume_2000
mean_SALT  = grid.integrate(ecco_ds.SALT * maskC_2000, ['X','Y','Z']).sum('tile').compute() / volume_2000

h0_0 = gsw.dynamic_enthalpy(ecco_ds.SALT*0+mean_SALT,mean_THETA,-ecco_ds.Z)
h0_zv_0 = gsw.dynamic_enthalpy(ecco_ds.SALT*0+mean_SALT,mean_THETA, H2/2)
h_zv_0 = h0_0 - h0_zv_0

dz_gb_2k    = grid.integrate((h_zv - h_zv_0) * maskC_2000 ,['Z']).compute() / H2 / g

In [10]:
# temperature contribution
h0 = gsw.dynamic_enthalpy(ecco_ds.SALT*0+mean_SALT, ecco_ds.THETA,-ecco_ds.Z)
h0_zv = gsw.dynamic_enthalpy(ecco_ds.SALT*0+mean_SALT, ecco_ds.THETA,H2/2)
h_zv = h0 - h0_zv

dz_gb_2k_Theta = grid.integrate((h_zv - h_zv_0) * maskC_2000 ,['Z']).compute() / H2 / g

In [11]:
# salinity contribution
h0 = gsw.dynamic_enthalpy(ecco_ds.SALT, ecco_ds.THETA*0+mean_THETA,-ecco_ds.Z)
h0_zv = gsw.dynamic_enthalpy(ecco_ds.SALT, ecco_ds.THETA*0+mean_THETA,H2/2)
h_zv = h0 - h0_zv

dz_gb_2k_Salin = grid.integrate((h_zv - h_zv_0) * maskC_2000 ,['Z']).compute() / H2 / g

In [12]:
dz_gb_2k = dz_gb_2k.rename("dz_gb_2k")
dz_gb_2k_Theta = dz_gb_2k_Theta.rename("dz_gb_2k_Theta")
dz_gb_2k_Salin = dz_gb_2k_Salin.rename("dz_gb_2k_Salin")
ds = xr.Dataset({"dz_gb_2k": dz_gb_2k, "dz_gb_2k_Theta": dz_gb_2k_Theta, "dz_gb_2k_Salin": dz_gb_2k_Salin})

In [13]:
#Local stratification, squared bouyancy frequency N2
maskW = (grid.diff(ecco_ds.maskC.astype(float),'Z',to='outer',boundary='extend')+1)

alpha = grid.interp(gsw.alpha(ecco_ds.SALT,ecco_ds.THETA,ecco_ds.Z)*ecco_ds.maskC,'Z',to='outer',boundary='extend')
beta  = grid.interp(gsw.beta (ecco_ds.SALT,ecco_ds.THETA,ecco_ds.Z)*ecco_ds.maskC,'Z',to='outer',boundary='extend')
N2_T = grid.interp(-g*alpha*grid.derivative(ecco_ds.THETA,'Z',to='outer',boundary='extend')*maskW , 'Z') * ecco_ds.maskC     #Interpolate again to get back on center point Z
N2_S = grid.interp( g*beta *grid.derivative(ecco_ds.SALT ,'Z',to='outer',boundary='extend')*maskW , 'Z') * ecco_ds.maskC
N2   = N2_T + N2_S

#Computing the depth-averaged value of N2 for depths shallower than 2000 meters
N2_2k = (grid.integrate(N2*maskC_2000,['Z']) / H2).compute()
N2_T_2k = (grid.integrate(N2_T*maskC_2000,['Z']) / H2).compute()
N2_S_2k = (grid.integrate(N2_S*maskC_2000,['Z']) / H2).compute()

ds['N2_2k'] = N2_2k
ds['N2_T_2k'] = N2_T_2k
ds['N2_S_2k'] = N2_S_2k

In [26]:
# surface fields
SST = ecco_ds.THETA.isel(k=0).compute()
SSS = ecco_ds.SALT.isel(k=0).compute()
SSD = gsw.sigma0(SSS, SST).compute()

ds['SST'] = SST
ds['SSS'] = SSS
ds['SSD'] = SSD

In [15]:
#Buoyancy content
density = gsw.rho(ecco_ds.SALT, ecco_ds.THETA,-ecco_ds.Z)
density_2k   = grid.integrate(density * maskC_2000 ,['Z']).compute()

#Ocean Heat Content
OHC_2k   = grid.integrate(ecco_ds.THETA * maskC_2000 ,['Z']).compute() * 4e6

#Ocean Salt Content
OSC_2k   = grid.integrate((ecco_ds.SALT-35) * maskC_2000 ,['Z']).compute()

ds['density_2k'] = density_2k
ds['OHC_2k'] = OHC_2k
ds['OSC_2k'] = OSC_2k

In [27]:
# Get the time coordinate as numeric values in years
time_numeric_years = np.arange(dz_gb_2k.sizes['time']) / 12  # Convert months to years

# Define a function to compute the linear trend
def compute_trend(data, time):
    # Use np.polyfit to compute the slope (trend)
    mask = ~np.isnan(data)                                    # Ignore NaN values
    if mask.sum() > 1:                                        # Ensure there are enough points for computation
        slope, _ = np.polyfit(time[mask], data[mask], 1)
        return slope
    else:
        return np.nan                                         # Return NaN if not enough data points

trend_dz_gb = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    dz_gb_2k*1000,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_dz_gb'] = trend_dz_gb

trend_zg_T = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    dz_gb_2k_Theta*1000,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_zg_T'] = trend_zg_T

trend_zg_S = xr.apply_ufunc(                    # Salinity trend
    compute_trend,
    dz_gb_2k_Salin*1000,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_zg_S'] = trend_zg_S

trend_N2 = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    N2_2k*1e8,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_N2'] = trend_N2

trend_N2_T = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    N2_T_2k*1e8,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_N2_T'] = trend_N2_T

trend_N2_S = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    N2_S_2k*1e8,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_N2_S'] = trend_N2_S

trend_buoyancy = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    -density_2k*g/rho0,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_buoyancy'] = trend_buoyancy

trend_OHC = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    OHC_2k,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_OHC'] = trend_OHC

trend_OSC = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    OSC_2k,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_OSC'] = trend_OSC

trend_SSD = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    SSD,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_SSD'] = trend_SSD

trend_SST = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    SST,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_SST'] = trend_SST

trend_SSS = xr.apply_ufunc(                     # Temperature trend 
    compute_trend,
    SSS,
    input_core_dims=[['time']],
    kwargs={'time': time_numeric_years},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)
ds['trend_SSS'] = trend_SSS

In [28]:
from scipy.stats import kendalltau

def kendalltau_pvalue(data, time_numeric):
    res = kendalltau(time_numeric,data)
    return res.pvalue

time_numeric = (dz_gb_2k_Theta.time - dz_gb_2k_Theta.time.min()).dt.total_seconds()

pvalue_dz = xr.apply_ufunc(
    kendalltau_pvalue,
    dz_gb_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_dz'] = pvalue_dz

pvalue_zg_T = xr.apply_ufunc(
    kendalltau_pvalue,
    dz_gb_2k_Theta,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_zg_T'] = pvalue_zg_T

pvalue_zg_S = xr.apply_ufunc(
    kendalltau_pvalue,
    dz_gb_2k_Salin,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_zg_S'] = pvalue_zg_S

pvalue_N2 = xr.apply_ufunc(
    kendalltau_pvalue,
    N2_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_N2'] = pvalue_N2

pvalue_N2_T = xr.apply_ufunc(
    kendalltau_pvalue,
    N2_T_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_N2_T'] = pvalue_N2_T

pvalue_N2_S = xr.apply_ufunc(
    kendalltau_pvalue,
    N2_S_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_N2_S'] = pvalue_N2_S

pvalue_buoyancy = xr.apply_ufunc(
    kendalltau_pvalue,
    -density_2k*g/rho0,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_buoyancy'] = pvalue_buoyancy

pvalue_OHC = xr.apply_ufunc(
    kendalltau_pvalue,
    OHC_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_OHC'] = pvalue_OHC

pvalue_OSC = xr.apply_ufunc(
    kendalltau_pvalue,
    OSC_2k,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_OSC'] = pvalue_OSC

pvalue_SSD = xr.apply_ufunc(
    kendalltau_pvalue,
    SSD,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_SSD'] = pvalue_SSD

pvalue_SST = xr.apply_ufunc(
    kendalltau_pvalue,
    SST,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_SST'] = pvalue_SST

pvalue_SSS = xr.apply_ufunc(
    kendalltau_pvalue,
    SSS,
    input_core_dims=[["time"]],
    kwargs={'time_numeric': time_numeric},
    vectorize=True,                       # Apply function across all grid points
    dask="parallelized",                  # Use Dask for parallel computation if zg is large
    output_dtypes=[float],
).compute()
ds['pvalue_SSS'] = pvalue_SSS


In [29]:
ds.to_netcdf(f'trend_stratification_2k_ECCO{ECCO_version}.nc')